# export

> Export a trained model to ONNX for production deployment, with a numerical check that ONNX Runtime reproduces the PyTorch output.

A dynamic batch axis is kept so the same file serves single-image and batched inference. The verification step catches op/opset mismatches before the model reaches a production line.

In [ ]:
#| default_exp export

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path

import numpy as np
import torch

## Export to ONNX

The dummy input must live on the same device as the model. We export with `dynamo=False` for a stable, widely-supported graph, then load the file back to run the structural checker.

In [ ]:
#| export
def export_onnx(
    model,                   # trained model returning logits
    output_path: str,        # destination .onnx path
    in_channels: int = 3,    # input channel count (must match training)
    input_size: int = 256,   # square input side for the export trace
    opset: int = 18,         # ONNX opset version
    verify: bool = True,     # compare ONNX Runtime output against PyTorch
) -> str:
    """Export `model` to ONNX with a dynamic batch axis; return the path."""
    import onnx
    model.eval()
    dev = next(model.parameters()).device
    dummy = torch.randn(1, in_channels, input_size, input_size, device=dev)
    with torch.no_grad():
        pt_out = model(dummy).cpu().numpy()
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        model, dummy, output_path,
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=opset, dynamo=False,
    )
    onnx.checker.check_model(onnx.load(output_path))
    if verify:
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(output_path)
            ort_out = sess.run(['logits'], {'image': dummy.cpu().numpy()})[0]
            max_diff = float(np.abs(pt_out - ort_out).max())
            assert max_diff < 1e-3, f'ONNX output diverges from PyTorch (max diff {max_diff})'
        except ImportError:
            pass  # onnxruntime not installed; structural check still ran
    return output_path

In [ ]:
# test: export a tiny 2-channel model and confirm a valid file is written
import tempfile
import torch.nn as nn
model = nn.Conv2d(2, 1, 3, padding=1)
with tempfile.TemporaryDirectory() as d:
    out = Path(d) / 'm.onnx'
    export_onnx(model, str(out), in_channels=2, input_size=32, verify=True)
    assert out.exists() and out.stat().st_size > 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()